## Computation of spherical harmonic coefficients


<a href="[text](https://creativecommons.org/licenses/by-nc/4.0/)">LaserIllumination</a> © 2025 by <a href="https://creativecommons.org">M. Cotelo, A. Lorca (Universidad Politécnica de Madrid)</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International</a><img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/by.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/nc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/sa.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;">


Authors:
- Manuel Cotelo Ferreiro (<manuel.cotelo@upm.es>) (Instituto de Fusión Nuclear Guillermo Velarde, Universidad Politécnica de Madrid)
- Alberto Lorca (<alberto.lorca@alumnos.upm.es>) (Universidad Politécnica de Madrid)

In [1]:
import numpy as np 
import pandas as pd 
from rwhist import compute_sph_coeffs, print_coeffs
from scipy.special import sph_harm

### Reading the histogram data

In [2]:
def load_histogram(filename, n_theta=50, n_phi=50, column="hs_density"):
    
    df = pd.read_hdf(filename)   # sin key="impactos"

    if column not in df.columns:
        raise KeyError(
            f"La columna '{column}' no existe. Columnas disponibles: {df.columns.tolist()}"
        )

    hist_2d = df[column].values.reshape((n_theta, n_phi))
    return hist_2d

hist = load_histogram("histograma_muestreo.h5", n_theta=50, n_phi=50, column="hs_density")


### Calculation of spherical harmonics

\begin{align}
a_{l,m} & = \int_{0}^{2 \pi} d \phi \int_{0}^{\pi} d \theta sin \theta f(\theta,\phi) Y_{l,m}^{*} (\theta,\phi) \\
& = \sum^{N} \int_{\phi_{j}}^{\phi_{j+1}} d \phi \int_{\theta_{i}}^{\theta_{i+1}} d \theta sin \theta f(\theta,\phi) Y_{l,m}^{*} (\theta,\phi) \\
\end{align}
 
Assume $f(\theta,\phi) = f_{i,j} = CTE$ for each sector $[\theta_{i},\theta_{i+1}] \times [\phi_{j},\phi_{j+1}]$
 
\begin{align}
a_{l,m} & = \sum^{N} f_{i,j} \int_{\phi_{j}}^{\phi_{j+1}} d \phi \int_{\theta_{i}}^{\theta_{i+1}} d \theta sin \theta Y_{l,m}^{*} (\theta,\phi) \\
& \simeq \sum^{N} f_{i,j} \cdot sin(\theta_{i + \frac{1}{2}}) \cdot Y_{l,m}^{*}(\theta_{i+\frac{1}{2}},\phi_{j+\frac{1}{2}}) \cdot S_{i,j}
\end{align}

### Physical approximation and calculation of spherical harmonics

In [3]:
def analyze_histogram(hist, theta_edges, phi_edges, Lmax=6):

    print("\n" + "="*80)
    print("=== ANALISIS DE ARMÓNICOS ESFÉRICOS DEL HISTOGRAMA REAL ===")
    print("="*80 + "\n")

    coeffs = compute_sph_coeffs(hist, theta_edges, phi_edges, Lmax)
    print_coeffs(coeffs)

    return coeffs


In [4]:

n_theta, n_phi = 50, 50
theta_edges = np.linspace(0, np.pi, n_theta + 1)
phi_edges = np.linspace(0, 2*np.pi, n_phi + 1)

# 4. Analizar armónicos
analyze_histogram(hist, theta_edges, phi_edges, Lmax=6)



=== ANALISIS DE ARMÓNICOS ESFÉRICOS DEL HISTOGRAMA REAL ===


================ COEFICIENTES NORMALIZADOS a_{lm} =================

  l    m        Re(a_lm)        Im(a_lm)          |a_lm|   |a_lm|/|a_00|
---------------------------------------------------------------------------
  0    0    2.820948e-01    0.000000e+00    2.820948e-01    1.000000e+00
  1   -1   -1.713621e-03   -8.317748e-03    8.492433e-03    3.010489e-02
  1    0   -5.416562e-02    0.000000e+00    5.416562e-02    1.920121e-01
  1    1    1.713621e-03   -8.317748e-03    8.492433e-03    3.010489e-02
  2   -2    7.147160e-02    3.418877e-02    7.922791e-02    2.808556e-01
  2   -1   -5.355559e-02   -1.133489e-02    5.474194e-02    1.940551e-01
  2    0    1.451657e-01    0.000000e+00    1.451657e-01    5.145992e-01
  2    1    5.355559e-02   -1.133489e-02    5.474194e-02    1.940551e-01
  2    2    7.147160e-02   -3.418877e-02    7.922791e-02    2.808556e-01
  3   -3   -1.182748e-02   -1.113744e-02    1.624598e-02    5.

{(0, 0): np.complex128(0.2820947917738782+0j),
 (1, -1): np.complex128(-0.0017136212146534146-0.008317747711907002j),
 (1, 0): np.complex128(-0.054165624603125934+0j),
 (1, 1): np.complex128(0.0017136212146534146-0.008317747711907002j),
 (2, -2): np.complex128(0.07147160161046827+0.034188769101468945j),
 (2, -1): np.complex128(-0.05355558610858767-0.011334890044513126j),
 (2, 0): np.complex128(0.14516574637893204+0j),
 (2, 1): np.complex128(0.05355558610858767-0.011334890044513126j),
 (2, 2): np.complex128(0.07147160161046827-0.034188769101468945j),
 (3, -3): np.complex128(-0.011827480978952337-0.011137439132289912j),
 (3, -2): np.complex128(-0.029969411191681738-0.011933756724256393j),
 (3, -1): np.complex128(0.009471074013854632-0.0018056143434248544j),
 (3, 0): np.complex128(-0.0022104185307980635+0j),
 (3, 1): np.complex128(-0.009471074013854632-0.0018056143434248544j),
 (3, 2): np.complex128(-0.029969411191681738+0.011933756724256393j),
 (3, 3): np.complex128(0.011827480978952337-